In [2]:
### hivis image scraper

# takes the master sheet as input with usgs site number, base image url, and interval type
# also needs a time frame

# creates datasheets for hivis sites 

import requests
import pandas as pd
from functools import reduce, partial
from tqdm.notebook import tqdm

#TODO add mastersheet logic
site_ids = [
    '02146409',    #lil sugar creek, Charlotte, NC
    '01462000',    #Delaware River at Lambertville NJ
]

base_image_url = [
    "https://usgs-nims-images.s3.amazonaws.com/720/NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte/NC_Little_Sugar_Creek_at_Medical_Center_Dr_at_Charlotte___{Y}-{M}-{D}T{h}-{m}-{s}Z.jpg",
    "https://usgs-nims-images.s3.amazonaws.com/720/NJ_Delaware_River_at_Lambertville_NJ/NJ_Delaware_River_at_Lambertville_NJ___{Y}-{M}-{D}T{h}-{m}-{s}Z.jpg",
]

intervals = [5, 60]

interval_type = {
    5: [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55],
    10: [0, 10, 20, 30, 40, 50],
    15: [0, 15, 30, 45],
    30: [0, 30],
    60: [0],
}

startDT = '2026-01-31T00:01'
endDT = '2026-01-31T02:59'

def make_spreadsheet(site_id, startDT, endDT, out_file='usgs_data.xlsx'):
    # get json from USGS API
    base_url = "https://waterservices.usgs.gov/nwis/iv/"
    params = {
        "format": 'json',
        "sites": site_id,
        "startDT": startDT,
        "endDT": endDT,
        "siteStatus": 'all',
    }
    r = requests.get(base_url, params=params)
    r.raise_for_status()
    data = r.json()
    
    time_series = data['value']['timeSeries']
    
    dfs = []
    for series in time_series:
        site_name = series["sourceInfo"]["siteName"]
        site_code = series["sourceInfo"]["siteCode"][0]["value"]
        variable = series["variable"]["variableName"]
        unit = series["variable"]["unit"]["unitCode"]
    
        values = series["values"][0]["value"]
        if not values:
            continue
    
        df = pd.DataFrame(values)
        df["dateTime"] = pd.to_datetime(df["dateTime"]).dt.tz_localize(None)
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    
        # Rename value column to variable name
        col_name = f"{variable} ({unit})"
        df = df[["dateTime", "value"]].rename(columns={"value": col_name})
    
        dfs.append(df)

    # Combine into one dataframe
    if not dfs:
        print(f"No data for site {site_id}")
        return
    
    full_df = reduce(
        lambda left, right: pd.merge(left, right, on="dateTime", how="outer"),
        dfs
    )

    full_df["siteName"] = site_name
    full_df["siteCode"] = site_code

    full_df = full_df.sort_values("dateTime")

    # Save to Excel
    full_df.to_excel(out_file, index=False)

def append_hivis_url(dt, index):    
    if dt.minute not in interval_type[intervals[index]]:
        return
        
    for seconds in range(60):
        request_url = base_image_url[index].format(
            Y=dt.year, 
            M=f"{dt.month:02d}", 
            D=f"{dt.day:02d}", 
            h=f"{dt.hour:02d}", 
            m=f"{dt.minute:02d}", 
            s=f"{seconds:02d}"
        )
    
        try:
            # Send the HTTP GET request
            response = requests.get(request_url, stream=True)
            
            # Check response status
            #print(f"📡 Server responded with status code: {response.status_code}")
            
            if response.status_code == 200:
                #print(f"got 200 for {request_url}")
                return request_url
            
        except requests.exceptions.RequestException as e:
            print("⚠️ An error occurred:")
            print(e)
    
# loop thru all sites
for i in range(len(site_ids)):
    # create datasheet with usgs measurements
    make_spreadsheet(site_ids[i], startDT, endDT, out_file = 'hivis_' + site_ids[i] + '_datasheet.xlsx')
    print(f"collected USGS data for site #{site_ids[i]}")
    
print("Collected USGS data for all sites.")

# find image url and append to spreadsheet
for i in range(len(site_ids)):
    df = pd.read_excel("hivis_" + site_ids[i] + "_datasheet.xlsx")
    datetime_col = "dateTime"
    url_col = "image_url"
    df[datetime_col] = pd.to_datetime(df[datetime_col])

    tqdm.pandas()
    df[url_col] = df[datetime_col].progress_apply(
        partial(append_hivis_url, index=i)
    )
    df = df[df[url_col].notna()]

    df.to_excel("hivis_" + site_ids[i] + "_datasheet.xlsx", index=False)
    print(f"collected {len(df)} image urls for site #{site_ids[i]}")
    
print("collected all image urls.")

collected USGS data for site #02146409
collected USGS data for site #01462000
Collected USGS data for all sites.


  0%|          | 0/35 [00:00<?, ?it/s]

collected 35 image urls for site #02146409


  0%|          | 0/11 [00:00<?, ?it/s]

collected 2 image urls for site #01462000
collected all image urls.
